# 🎙️ Bangla Chatterbox GPU Voice Cloning Server for Movie Dubbing

This Kaggle notebook runs **Bangla Chatterbox** on a free Kaggle GPU (NVIDIA T4 x2 or P100) and exposes a free, secure public API via Cloudflare Tunnel. Your local **Movie Dubbing** application connects to this endpoint to clone character voices into natural Bengali with high fidelity.

### ⚡ Quick Start:
1. **Enable GPU**: On the right sidebar, open **Settings** -> **Accelerator** -> select **GPU T4 x2** (or GPU P100).
2. **Enable Internet**: Ensure **Internet** toggle is turned **ON** in the right sidebar.
3. **Run Cell 1**: Installs dependencies and downloads the Bangla Chatterbox weights.
4. **Run Cell 2**: Launches the API server and prints your public `https://xxxx.trycloudflare.com` URL.
5. **Copy URL**: Copy that URL into your local Movie Dubbing app (or set `KAGGLE_CHATTERBOX_URL` in your terminal).

In [ ]:
# Cell 1: Environment & Dependency Setup
!pip install -q chatterbox-tts fastapi uvicorn pycloudflared python-multipart torchaudio nest-asyncio

# Clone fine-tuned Bangla Chatterbox model repository from Hugging Face
import os
if not os.path.exists('/kaggle/working/chatterbox-bangla-tts'):
    print('Cloning Banglabox/chatterbox-bangla-tts...')
    !git clone https://huggingface.co/Banglabox/chatterbox-bangla-tts /kaggle/working/chatterbox-bangla-tts
    if os.path.exists('/kaggle/working/chatterbox-bangla-tts/inference/requirements.txt'):
        !pip install -q -r /kaggle/working/chatterbox-bangla-tts/inference/requirements.txt

print('Setup complete! Ready to start GPU server.')

In [ ]:
# Cell 2: Start Bangla Chatterbox Server & Cloudflare Public Tunnel
import io, os, sys, time, threading, tempfile, zipfile
from pathlib import Path
from typing import Any

import torch, torchaudio, uvicorn
from fastapi import FastAPI, File, Form, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response

app = FastAPI(title='Bangla Chatterbox API')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_credentials=True, allow_methods=['*'], allow_headers=['*'])

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
REF_CACHE_DIR = Path('/tmp/chatterbox_speaker_refs')
REF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

MODEL, MODEL_TYPE, MODEL_SR = None, 'none', 24_000

def init_model():
    global MODEL, MODEL_TYPE, MODEL_SR
    print(f'Device: {DEVICE}')
    repo = Path('/kaggle/working/chatterbox-bangla-tts')
    if (repo / 'inference' / 'infer.py').exists() or (repo / 'infer.py').exists():
        try:
            sys.path.insert(0, str(repo / 'inference'))
            sys.path.insert(0, str(repo))
            from infer import BanglaTTS
            MODEL = BanglaTTS(str(repo))
            MODEL_TYPE = 'Banglabox/chatterbox-bangla-tts'
            print('Loaded Banglabox pipeline!')
            return
        except Exception as e:
            print('Banglabox load note:', e)
    try:
        from chatterbox.tts import ChatterboxTTS
        MODEL = ChatterboxTTS.from_pretrained(device=DEVICE)
        MODEL_TYPE = 'chatterbox-tts'
        MODEL_SR = getattr(MODEL, 'sr', 24_000)
        print('Loaded ChatterboxTTS baseline!')
    except Exception as e:
        print('Standard chatterbox load note:', e)

init_model()

@app.get('/health')
def health():
    vram = 0.0
    if DEVICE == 'cuda':
        free, total = torch.cuda.mem_get_info()
        vram = round(free / (1024**3), 2)
    return {'status': 'ok', 'model': MODEL_TYPE, 'device': DEVICE, 'vram_free_gb': vram}

@app.post('/upload_reference')
async def upload_ref(speaker_id: str = Form(...), reference_file: UploadFile = File(...)):
    content = await reference_file.read()
    (REF_CACHE_DIR / f'spk_{speaker_id}.wav').write_bytes(content)
    return {'status': 'cached', 'speaker_id': speaker_id}

def synth_audio(text: str, ref_path: str | None = None):
    if MODEL_TYPE == 'Banglabox/chatterbox-bangla-tts':
        res = MODEL.synthesize(text, ref_audio=ref_path)
        return (res if isinstance(res, torch.Tensor) else torch.tensor(res)), MODEL_SR
    elif 'chatterbox' in MODEL_TYPE:
        wav = MODEL.generate(text, audio_prompt_path=str(ref_path)) if ref_path else MODEL.generate(text)
        return wav, MODEL_SR
    sr = 24_000
    t = torch.linspace(0, max(1.0, len(text)*0.08), int(sr * max(1.0, len(text)*0.08)))
    return (0.1 * torch.sin(2 * 3.14159 * 220 * t)).unsqueeze(0), sr

@app.post('/synthesize')
async def synthesize(text: str = Form(...), speaker_id: str = Form('0'), reference_audio: UploadFile | None = File(None)):
    ref_path = None
    if reference_audio:
        content = await reference_audio.read()
        if len(content) > 0:
            p = REF_CACHE_DIR / f'spk_{speaker_id}.wav'
            p.write_bytes(content)
            ref_path = str(p)
    if not ref_path and (REF_CACHE_DIR / f'spk_{speaker_id}.wav').exists():
        ref_path = str(REF_CACHE_DIR / f'spk_{speaker_id}.wav')

    wav, sr = synth_audio(text.strip(), ref_path=ref_path)
    if wav.ndim == 1: wav = wav.unsqueeze(0)
    buf = io.BytesIO()
    torchaudio.save(buf, wav.cpu(), sr, format='wav')
    return Response(content=buf.getvalue(), media_type='audio/wav')

from pycloudflared import try_cloudflare
def tunnel():
    time.sleep(2)
    pub = try_cloudflare(port=8000)
    print('\n' + '='*65)
    print('🚀 YOUR BANGLA CHATTERBOX SERVER IS LIVE!')
    print(f'🌐 Public API URL: {pub.tunnel}')
    print('='*65)
    print('👉 Copy this URL into Movie Dubbing or set: $env:KAGGLE_CHATTERBOX_URL="' + str(pub.tunnel) + '"\n')

threading.Thread(target=tunnel, daemon=True).start()
config = uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='info')
server = uvicorn.Server(config)
await server.serve()
